# Notebook-3

The goal of this notebook is to answer:
- Does when a student engages matter more than how much they engage? **means** which factor is more impactful on student outcomes: Timing or Frequency/Volume

# What We Will (and Will NOT) Do

### We WILL:
- Use date (including negative values)
- Re-introduce temporal data only where needed
- Create early-warning features usable in production

### We WILL NOT:
- Keep raw 10M rows
- Build sequences yet
- Add deep models

In [1]:
import pandas as pd
import gc 
from pathlib import Path

In [2]:
try :
    root_folder_path = Path(__file__).resolve().parent.parent.parent
except:
    root_folder_path = Path().resolve().parent.parent.parent

print(root_folder_path)

D:\AI-ML\REAL WORLD PROJECTS


In [4]:
base_path = Path(root_folder_path) / "MACHINE LEARNING/Academic-Risk-Engagement-Prediction-System/Dataset"
base_path_raw=Path(root_folder_path)/ "MACHINE LEARNING/Academic-Risk-Engagement-Prediction-System/data"

# Load Required Datasets

In [ ]:
student_df=pd.read_csv(base_path /'student_master_v2.csv')

# Reload studentVle dataset

In [9]:
student_vle = pd.read_csv(base_path_raw / "studentVle.csv"
            , usecols = ['id_student', 'date', 'sum_click'],
            dtype={
                "id_student" : "int32",
                "date" : "int16",
                "sum_click" : "int16"
            })

# Understanding date properly

(Only concept, no code here)

Why does date have negative values?
- date = date relative to course start
- Negative -> pre-course engagement

This reflects:
- curiosty
- preparation
- proactive behaviour

Im research, this is a strong success indicator.



In [10]:
student_vle.head()

,id_student,date,sum_click
0,28400,-10,4
1,28400,-10,1
2,28400,-10,1
3,28400,-10,11
4,28400,-10,1


In [11]:
first_activity=(
    student_vle
    .groupby('id_student')['date']
    .min()
    .reset_index()
    .rename(columns={'date':'first_activity_day'})
)

In [12]:
first_activity.head()

,id_student,first_activity_day
0,6516,-23
1,8462,-6
2,11391,-5
3,23629,-6
4,23698,-18


In [13]:
first_activity.min()

id_student            6516
first_activity_day     -25
dtype: int32

In [14]:
first_activity.max()

id_student            2698588
first_activity_day        238
dtype: int32

# pre-course engagement flag

In [15]:
pre_course_engaged=(
    student_vle
    .assign(pre_course=lambda x:x['date']<0)
    .groupby("id_student")['pre_course']
    .any()
    .astype(int)
    .reset_index(name='pre_course_engaged')
)

In [16]:
pre_course_engaged.head()

,id_student,pre_course_engaged
0,6516,1
1,8462,1
2,11391,1
3,23629,1
4,23698,1


# Early phase engagement (First 14 days)

Day 0 --> Day 1

In [17]:
early_engagement=(
    student_vle
    .query('date>=0 and date<=14')
    .groupby('id_student')
    .agg(
        early_click=('sum_click','sum'),
        early_active_days=('date','nunique')
    )
.reset_index()
)

In [18]:
early_engagement.head()

,id_student,early_click,early_active_days
0,6516,292,11
1,8462,243,10
2,11391,203,5
3,23629,28,3
4,23698,165,10


# Clean up memory

In [19]:
del student_vle
gc.collect()

14

# Merge all temporal features

In [20]:
student_temporal=(
    student_df
    .merge(first_activity,on='id_student',how='left')
    .merge(pre_course_engaged,on='id_student',how='left')
    .merge(early_engagement,on='id_student',how='left')
)

In [21]:
student_temporal.head()

,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,avg_clicks_per_day,max_clicks_day,active_days,sucess,engagement_level,no_engagement_flag,first_activity_day,pre_course_engaged,early_click,early_active_days
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,4.765306,76.0,40.0,1,Medium,0,-5.0,1.0,203.0,5.0
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,3.337209,23.0,80.0,1,High,0,-10.0,1.0,241.0,7.0
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,3.697368,23.0,12.0,0,Low,0,-10.0,1.0,179.0,6.0
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,3.254902,22.0,123.0,1,High,0,-10.0,1.0,180.0,8.0
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,2.937500,22.0,70.0,1,Medium,0,-10.0,1.0,177.0,9.0


In [22]:
student_temporal.info()

<class 'pandas.DataFrame'>
RangeIndex: 32593 entries, 0 to 32592
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   code_module           32593 non-null  str    
 1   code_presentation     32593 non-null  str    
 2   id_student            32593 non-null  int64  
 3   gender                32593 non-null  str    
 4   region                32593 non-null  str    
 5   highest_education     32593 non-null  str    
 6   imd_band              31482 non-null  str    
 7   age_band              32593 non-null  str    
 8   num_of_prev_attempts  32593 non-null  int64  
 9   studied_credits       32593 non-null  int64  
 10  disability            32593 non-null  str    
 11  final_result          32593 non-null  str    
 12  total_click           32593 non-null  float64
 13  avg_clicks_per_day    32593 non-null  float64
 14  max_clicks_day        32593 non-null  float64
 15  active_days           32593 no

In [23]:
student_temporal['pre_course_engaged']=student_temporal['pre_course_engaged'].fillna(0)

student_temporal[['early_click','early_active_days']]=(student_temporal[['early_click','early_active_days']].fillna(0))

# Answer the core question

Does early engagement matter?

In [24]:
student_temporal.groupby('sucess')[[
    'first_activity_day','early_click','pre_course_engaged'
]].mean()

,first_activity_day,early_click,pre_course_engaged
sucess,,,
0,-8.175360,105.372036,0.667306
1,-9.968535,207.136432,0.866948


# Save final feature set

In [27]:
student_temporal.to_csv(f"{Path(root_folder_path) / "MACHINE LEARNING/Academic-Risk-Engagement-Prediction-System/Dataset/student_master_v3.csv"}",index=False)